In [ ]:
import pandas as pd

In [ ]:
sales = pd.read_csv('vendas_2023_2024.csv')
products = pd.read_csv('produtos_raw.csv')
costs = pd.read_json('custos_importacao.json')
customers = pd.read_json('clientes_crm.json')
exchange_rates = pd.read_csv('bcdata.sgs.csv', sep=';')

### Tabela de Clientes (Customers Table)

In [ ]:
customers.head()

,full_name,location,code,email
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",1,femininos.oliveira.antunes@icloud.com
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",2,nunes.fernanda.soares.azevedo.vieira@outlook.com
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro#gmail.com
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira#gmail.com
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas#icloud.com


In [ ]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   full_name  49 non-null     object
 1   location   49 non-null     object
 2   code       49 non-null     int64 
 3   email      49 non-null     object
dtypes: int64(1), object(3)
memory usage: 1.7+ KB


In [ ]:
# 1. Validação e limpeza de localizações
# 1. Location data validation and cleaning

customers['location'].unique()

array(['Aratu (Candeias) , BA', 'PE , Recife', 'Rio Grande,RS',
       'AC , Rio Branco', 'PA - Santarém Novo',
       'Fortaleza do Tabocão , TO', 'PB/Cabedelo', 'SE - Aracaju',
       'PB - João Pessoa', 'Santarém / PA', 'BA - Porto Seguro',
       'TO , Fortaleza do Tabocão', 'PA / Santarém', 'AM , Itacoatiara',
       'Fortaleza do Tabocão,TO', 'Fortaleza,CE', 'MS - Corumbá',
       'Santarém - PA', 'Maceió / AL', 'PA , Santarém Novo',
       'AC,Rio Branco', 'SE / Aracaju', 'Santos - SP', 'Laguna / SC',
       'ES / São Mateus', 'Manaus/AM', 'Salvador,BA', 'PR , Antonina',
       'CE,Fortaleza', 'Itacoatiara - AM', 'São Luís,MA', 'Rio Grande/RS',
       'Fortaleza / CE', 'João Pessoa / PB', 'Belém/PA',
       'MA , Itaqui (São Luís)', 'TO/Fortaleza do Tabocão',
       'PE / Suape (Ipojuca)', 'Corumbá,MS', 'BA / Aratu (Candeias)',
       'Vila do Conde (Barcarena) - PA', 'MA/Itaqui (São Luís)',
       'Paranaguá / PR', 'Santarém,PA', 'Macapá/AP', 'RJ / Niterói',
       'PE - Suape 

In [ ]:
# 2. Padronização de dados com Regex
# 2. Data standardization with Regex

customers['client_state'] = customers['location'].str.extract(r'\b([A-Z]{2})\b') # Extração dos Estados

customers['client_city'] = (customers['location']
  .str.replace(r'\b[A-Z]{2}\b', '', regex=True)
  .str.replace(r'[,/\-]', '', regex=True)
  .str.strip()
  .str.title()
 )

customers[['location', 'client_state', 'client_city']].head()

,location,client_state,client_city
0,"Aratu (Candeias) , BA",BA,Aratu (Candeias)
1,"PE , Recife",PE,Recife
2,"Rio Grande,RS",RS,Rio Grande
3,"AC , Rio Branco",AC,Rio Branco
4,PA - Santarém Novo,PA,Santarém Novo


In [ ]:
# 3. Formatação de endereços de e-mail
# 3. Email address formatting

customers['email'] = customers['email'].str.replace('#', '@', regex=False)

customers['email'].sort_values()

,email
31,antunes.coelho.batista.teixeira.daniela@hotmai...
29,araújo.tavares.azevedo.victor.correia@gmail.com
21,borges.vieira.daniela.mendonça.farias@protonma...
5,coelho.pinheiro.peixoto.antônia.cavalcanti@aol...
40,costa.correia.ribeiro.santos.pedro@yahoo.com
45,costa.farias.coelho.ana.silva@hotmail.com
16,costa.paiva.luís.coelho.cardoso@aol.com
48,costa.tavares.antunes.diego@icloud.com
42,farias.jéssica.cunha@outlook.com
2,farias.teixeira.daniel.ribeiro@gmail.com


In [ ]:
# 4. Verificação de valores nulos
# 4. Checking for null values

customers.isna().sum()

,0
full_name,0
location,0
code,0
email,0
client_state,0
client_city,0


In [ ]:
# 5. Verificação de registros duplicados
# 5. Checking for duplicate records

print(f'Total Dataframe rows: {len(customers)}')
print(f'Total duplicates: {customers.duplicated().sum()}')
print(f'Total id duplicates: {customers.duplicated(subset=['code']).sum()}')

Total Dataframe rows: 49
Total duplicates: 0
Total id duplicates: 0


In [ ]:
# 6. Reorganização e renomeação de colunas
# 6. Column reordering and renaming

customers = customers.rename(columns={

      # Formato [Entidade]_[Atributo] para adequação ao padrão de mercado
      # [Entity]_[Attribute] format for market standard alignment

    'code': 'client_id',
    'full_name': 'client_name',
    'email': 'client_email'
})

customers = customers.drop(columns=['location'])

customers = customers[['client_id', 'client_name', 'client_email', 'client_state', 'client_city']]

customers.head()

,client_id,client_name,client_email,client_state,client_city
0,1,Femininos Oliveira Antunes,femininos.oliveira.antunes@icloud.com,BA,Aratu (Candeias)
1,2,Fernanda Azevedo Soares Nunes Vieira,nunes.fernanda.soares.azevedo.vieira@outlook.com,PE,Recife
2,3,Daniel Farias Ribeiro Teixeira,farias.teixeira.daniel.ribeiro@gmail.com,RS,Rio Grande
3,4,Thiago Moreira,thiago.moreira@gmail.com,AC,Rio Branco
4,5,Pedro Freitas,pedro.freitas@icloud.com,PA,Santarém Novo


### Tabela de Produtos (Products Table)

In [ ]:
products.head()

,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz


In [ ]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   name             157 non-null    object
 1   price            157 non-null    object
 2   code             157 non-null    int64 
 3   actual_category  157 non-null    object
dtypes: int64(1), object(3)
memory usage: 5.0+ KB


In [ ]:
# 1. Conferindo as diferentes nomenclaturas das Categorias
# 1. Checking Category naming variations

products['actual_category'].unique()

array(['ELETRONICOS', 'E L E T R Ô N I C O S', 'Eletrunicos',
       'Eletronicoz', 'eLeTrÔnIcOs', 'eletrônicos', 'Eletrônicos',
       'Eletroniscos', 'Eletronicos', 'eletronicos', 'EletrônicoS',
       'ELEtRÔNICOS', 'PROPULSAO', 'Propulção', 'Prop', 'Propulssão',
       'propulsao', 'P R O P U L S Ã O', 'Propução', 'propulsão',
       'pRoPuLsÃo', 'Propulçao', 'Propulsam', 'PrOpUlSãO', 'Ancoragem',
       'AnCoRaGeM', 'Encoragem', 'Ancoraguem', 'Ancorajm', 'AncorageM',
       'A N C O R A G E M', 'ANCORAGEM', 'aNcOrAgEm', 'Ancorajem',
       'Encoragi', 'ancoragem', 'Ancorajen', 'AncorajeM', 'Ancoragen'],
      dtype=object)

In [ ]:
# 2. Padronização das nomenclaturas com Regex
# 2. Standardization of naming conventions via Regex

products['actual_category'] = products['actual_category'].str.replace(' ', '').str.lower()

mapeamento = {
    r'.*eletr.*': 'eletrônicos',
    r'.*ncor.*': 'ancoragem',
    r'.*prop.*': 'propulsão'
}

products['actual_category'] = products['actual_category'].replace(mapeamento, regex=True)

  #Checagem dos novos valores
  #Checking new values
products['actual_category'].unique()

array(['eletrônicos', 'propulsão', 'ancoragem'], dtype=object)

In [ ]:
# 3. Conversão dos valores de "Price" para o tipo numérico
# 3. Converting "Price" values to numeric data types

products['list_price_BRL'] = products['price'].str.removeprefix('R$ ')

products['list_price_BRL'] = pd.to_numeric(products['list_price_BRL'], errors='coerce') # o código "erros=coerce" retornará NaN para os valores que não forem convertidos para numéricos

print(f'Column new type: {products['list_price_BRL'].dtypes}') # Checagem da conversão

Column new type: float64


In [ ]:
# 4. Conferindo colunas com valores nulos
# 4. Checking columns for null values

products.isna().sum()

,0
name,0
price,0
code,0
actual_category,0
list_price_BRL,0


In [ ]:
# 5. Conferindo a quantidade de duplicatas
# 5. Checking for duplicate records

print(f'Total Dataframe rows: {len(products)}')
print(f'Total duplicates: {products.duplicated().sum()}')
print(f'Total code duplicates: {products.duplicated(subset=['code']).sum()}')
print(f'Total product name duplicates: {products.duplicated(subset=['name']).sum()}')


Total Dataframe rows: 157
Total duplicates: 7
Total code duplicates: 7
Total product name duplicates: 7


In [ ]:
# 6. Conferindo as linhas duplicadas
# 6. Inspecting duplicate rows

conflitos = products[products.duplicated(subset=['code'], keep=False)]
conflitos.sort_values(by='code')

,name,price,code,actual_category,list_price_BRL
36,GPS Lowrance Evo Storm Drift,R$ 6067.71,37,eletrônicos,6067.71
37,GPS Lowrance Evo Storm Drift,R$ 6067.71,37,eletrônicos,6067.71
62,Motor Diesel Yanmar Velocity 37HP,R$ 102221.97,62,propulsão,102221.97
63,Motor Diesel Yanmar Velocity 37HP,R$ 102221.97,62,propulsão,102221.97
64,Motor Diesel Yanmar Velocity 37HP,R$ 102221.97,62,propulsão,102221.97
65,Motor Diesel Yanmar Velocity 37HP,R$ 102221.97,62,propulsão,102221.97
131,Cabo de Nylon Delta Velocity Core Mako,R$ 1549.35,127,ancoragem,1549.35
132,Cabo de Nylon Delta Velocity Core Mako,R$ 1549.35,127,ancoragem,1549.35
124,Boia de Arqueamento Delta Nexus,R$ 4349.86,145,ancoragem,4349.86
150,Boia de Arqueamento Delta Nexus,R$ 4349.86,145,ancoragem,4349.86


In [ ]:
# 7. Removendo as duplicatas
# 7. Removing duplicate records

products = products.drop_duplicates(subset=['code'], keep='first')

print(f'Total rows in the new Dataframe: {len(products)}')

Total rows in the new Dataframe: 150


In [ ]:
# 8. Reorganização e renomeação de colunas
# 8. Column reordering and renaming

products = products.rename(columns={

      # Formato [Entidade]_[Atributo] para adequação ao padrão de mercado
      # [Entity]_[Attribute] format for market standard alignment

    'code': 'product_id',
    'name': 'product_name',
    'actual_category': 'product_category'
})

products = products.drop(columns='price')

products = products[['product_id', 'product_name', 'product_category', 'list_price_BRL']]

products.head()

,product_id,product_name,product_category,list_price_BRL
0,1,Transponder AIS Maré Magnum,eletrônicos,33122.52
1,2,Transponder Furuno Marlin,eletrônicos,13998.15
2,3,Radar Furuno Pulse Leviathan,eletrônicos,9024.19
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,3381.88
4,5,Piloto Automático Furuno Storm,eletrônicos,23669.01


### Tabela de Vendas (Sales Table)

In [ ]:
sales.head()

,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,15-09-2024
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


In [ ]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          9895 non-null   int64  
 1   id_client   9895 non-null   int64  
 2   id_product  9895 non-null   int64  
 3   qtd         9895 non-null   int64  
 4   total       9895 non-null   float64
 5   sale_date   9895 non-null   object 
dtypes: float64(1), int64(4), object(1)
memory usage: 464.0+ KB


In [ ]:
# 1. Formatação da coluna de datas para o formato ISO (AAAA-MM-DD)
# 1. Formatting date columns to ISO 8601 standard (YYYY-MM-DD)

sales['sale_date'] = pd.to_datetime(sales['sale_date'], format = 'mixed', errors='coerce')

  # Checagem da conversão
  # Conversion check
print(f'The new type of the sale_date column is: {sales['sale_date'].dtype} \nNew format: {sales['sale_date'].max()}')

The new type of the sale_date column is: datetime64[ns] 
New format: 2024-12-31 00:00:00


In [ ]:
# 2. Conferindo datas faltantes
# 2. Checking missing dates

total_calendar = pd.date_range(start=sales['sale_date'].min(), end=sales['sale_date'].max(), freq='D')
missing_dates = total_calendar.difference(sales['sale_date'])

print(f'Number of missing days: {len(missing_dates)}')
print(f'Missing dates are: {missing_dates}')

Number of missing days: 5
Missing dates are: DatetimeIndex(['2023-01-15', '2023-03-21', '2023-07-30', '2023-08-14',
               '2024-05-14'],
              dtype='datetime64[ns]', freq=None)


In [ ]:
# 3. Completando a tabela com as datas faltantes
# 3. Completing the table with missing dates

total_calendar_df = pd.DataFrame({'sale_date': total_calendar})

sales = pd.merge(total_calendar_df, sales, on='sale_date', how='left')

sales[['total', 'qtd']] = sales[['total', 'qtd']].fillna(0)

  # Checagem das novas colunas
  # Checking new columns
sales[sales['sale_date'].isin(['2023-01-15', '2023-03-21', '2023-07-30', '2023-08-14', '2024-05-14'])]

/tmp/ipykernel_5335/2594009487.py:12: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  sales[sales['sale_date'].isin(['2023-01-15', '2023-03-21', '2023-07-30', '2023-08-14', '2024-05-14'])]


,sale_date,id,id_client,id_product,qtd,total
219,2023-01-15,NaN,NaN,NaN,0.0,0.0
1095,2023-03-21,NaN,NaN,NaN,0.0,0.0
2901,2023-07-30,NaN,NaN,NaN,0.0,0.0
3086,2023-08-14,NaN,NaN,NaN,0.0,0.0
6788,2024-05-14,NaN,NaN,NaN,0.0,0.0


In [ ]:
# 4. Conversão dos valores monetários
# 4. Converting monetary values

  # o código "erros=coerce" retornará NaN para os valores que não forem convertidos para numéricos
  # the code "erros=coerce" will return NaN for values ​​that are not converted to numeric
sales['total'] = pd.to_numeric(sales['total'], errors='coerce')

  # Checagem da conversão
  # Conversion check
print(f'Column new type: {sales['total'].dtypes}')

Column new type: float64


In [ ]:
# 5. Conversão dos valores inteiros
# 5. Conversion of integer values

sales[['id', 'id_client', 'id_product', 'qtd']] = sales[['id', 'id_client', 'id_product', 'qtd']].astype('Int64')

  # Checagem da conversão
  # Conversion check
print(f'Column new type: {sales[['id', 'id_client', 'id_product', 'qtd']].dtypes}')

Column new type: id            Int64
id_client     Int64
id_product    Int64
qtd           Int64
dtype: object


In [ ]:
# 5. Conferindo colunas com valores nulos
# 5. Checking columns for null values

  # Os valores nulos correspondem aos novos registros de datas feitas no passo 3. Portanto, não é necessário a sua remoção.
  # Null values ​​correspond to new date records made in step 3. Therefore, no removal is required.

sales.isna().sum()

,0
sale_date,0
id,5
id_client,5
id_product,5
qtd,0
total,0


In [ ]:
# 6. Conferindo a quantidade de duplicatas
# 6. Checking the count of duplicate records

  # Os valores duplicados correspondem aos registros de datas sem vendas.
  # Esses registros serão utilizados futuramente para o cálculo da média de vendas, portanto, não é necessário a sua remoção.

  # The duplicate values ​​correspond to date records with no sales.
  # These records will be used in the future to calculate the sales average; therefore, their removal is not necessary.

print(f'Total Dataframe rows: {len(sales)}')
print(f'Total duplicates: {sales.duplicated().sum()}')
print(f'Total id duplicates: {sales.duplicated(subset=['id']).sum()}')

duplicates_sales = sales[sales.duplicated(subset=['id'], keep='first')]
display(duplicates_sales)

Total Dataframe rows: 9900
Total duplicates: 0
Total id duplicates: 4


,sale_date,id,id_client,id_product,qtd,total
1095,2023-03-21,<NA>,<NA>,<NA>,0,0.0
2901,2023-07-30,<NA>,<NA>,<NA>,0,0.0
3086,2023-08-14,<NA>,<NA>,<NA>,0,0.0
6788,2024-05-14,<NA>,<NA>,<NA>,0,0.0


In [ ]:
# 7. Reorganização e renomeação de colunas
# 7. Column reordering and renaming

sales = sales.rename(columns={

    # Formato [Entidade]_[Atributo] para adequação ao padrão de mercado
    # [Entity]_[Attribute] format for market standard alignment

    'id': 'sale_id',
    'id_client': 'client_id',
    'id_product': 'product_id',
    'qtd': 'product_qty',
    'total': 'sale_total_BRL'
})


cols = list(sales.columns)
cols.insert(4, cols.pop(cols.index('sale_date')))

sales = sales[cols]


sales.head()

,sale_id,client_id,product_id,product_qty,sale_date,sale_total_BRL
0,666,14,15,5,2023-01-01,132524.05
1,1230,17,91,4,2023-01-01,512566.80
2,2300,30,95,9,2023-01-01,596858.40
3,3131,28,130,13,2023-01-01,53873.00
4,3549,10,53,13,2023-01-01,662886.25


### Tabela de Custos (Costs Table)

In [ ]:
costs.head()

,product_id,product_name,category,historic_data
0,1,Transponder AIS Maré Magnum,eletrônicos,"[{'start_date': '10/08/2016', 'usd_price': 105..."
1,2,Transponder Furuno Marlin,eletrônicos,"[{'start_date': '23/11/2017', 'usd_price': 432..."
2,3,Radar Furuno Pulse Leviathan,eletrônicos,"[{'start_date': '12/04/2016', 'usd_price': 254..."
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,"[{'start_date': '04/03/2016', 'usd_price': 909..."
4,5,Piloto Automático Furuno Storm,eletrônicos,"[{'start_date': '10/02/2016', 'usd_price': 600..."


In [ ]:
costs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_id     150 non-null    int64 
 1   product_name   150 non-null    object
 2   category       150 non-null    object
 3   historic_data  150 non-null    object
dtypes: int64(1), object(3)
memory usage: 4.8+ KB


In [ ]:
# 1. Checagem dos valores da coluna historic_data
# 1. Checking the values in the historic_data column

print(f'Costs table first line: {costs.loc[0, 'historic_data']}')

Costs table first line: [{'start_date': '10/08/2016', 'usd_price': 10583.63}, {'start_date': '15/06/2018', 'usd_price': 8778.36}, {'start_date': '25/09/2018', 'usd_price': 8023.87}, {'start_date': '19/03/2019', 'usd_price': 8772.78}, {'start_date': '17/01/2020', 'usd_price': 7918.18}, {'start_date': '17/06/2020', 'usd_price': 6310.01}, {'start_date': '02/07/2021', 'usd_price': 6586.7}, {'start_date': '16/05/2022', 'usd_price': 6538.2}, {'start_date': '28/02/2023', 'usd_price': 6360.91}, {'start_date': '17/10/2023', 'usd_price': 6574.8}, {'start_date': '16/02/2024', 'usd_price': 6657.12}, {'start_date': '22/02/2024', 'usd_price': 6703.2}, {'start_date': '15/03/2024', 'usd_price': 6633.66}, {'start_date': '02/08/2024', 'usd_price': 5774.5}, {'start_date': '08/04/2025', 'usd_price': 5579.75}]


In [ ]:
# 2. Separação dos valores de historic_data
# 2. Splitting and exploding historic_data values

costs_exploded = costs.explode('historic_data').reset_index(drop=True)

print(f'New costs table first line: {costs_exploded.loc[0, 'historic_data']}')

New costs table first line: {'start_date': '10/08/2016', 'usd_price': 10583.63}


In [ ]:
# 3. Normalização da coluna historic_data
# 3. Normalizing the 'historic_data' column

costs_normalization = pd.json_normalize(costs_exploded['historic_data'])

costs_normalization.head()

,start_date,usd_price
0,10/08/2016,10583.63
1,15/06/2018,8778.36
2,25/09/2018,8023.87
3,19/03/2019,8772.78
4,17/01/2020,7918.18


In [ ]:
# 4. Junção das colunas normalizadas com o restante da tabela
# 4. Joining normalized columns with the main dataset

  # axix=1 para concatenação no sentido das colunas
  # axix=1 for column-wise concatenation
costs = pd.concat([costs_exploded.drop(columns=['historic_data']), costs_normalization], axis=1)

costs.head()

,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,10/08/2016,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,15/06/2018,8778.36
2,1,Transponder AIS Maré Magnum,eletrônicos,25/09/2018,8023.87
3,1,Transponder AIS Maré Magnum,eletrônicos,19/03/2019,8772.78
4,1,Transponder AIS Maré Magnum,eletrônicos,17/01/2020,7918.18


In [ ]:
# 5. Formatação da coluna de datas para o formato ISO (AAAA-MM-DD)
# 5. Formatting date columns to ISO 8601 standard (YYYY-MM-DD)

costs['start_date'] = pd.to_datetime(costs['start_date'], format = 'mixed', errors='coerce')

   # Checagem da conversão
   # Conversion check
print(f'start_date column new type: {costs['start_date'].dtype} \nNew format: {costs['start_date'].max()}')

start_date column new type: datetime64[ns] 
New format: 2025-12-31 00:00:00


In [ ]:
# 6. Conversão de moeda com ajuste de câmbio por data (Lookback)
# 6. Currency conversion with Date-Based exchange rate (Lookback)

if exchange_rates['valor'].dtype == 'object':
    exchange_rates['valor'] = exchange_rates['valor'].str.replace(',', '.').astype(float)

exchange_rates['data'] = pd.to_datetime(exchange_rates['data'], dayfirst=True)

costs = costs.sort_values('start_date')
exchange_rates = exchange_rates.sort_values('data')

costs = pd.merge_asof(
    costs,
    exchange_rates,
    left_on='start_date',
    right_on='data',
    direction='backward'
)

costs['brl_price'] = costs['usd_price'] * costs['valor']

In [ ]:
# 7. Conversão dos valores monetários
# 7. Converting monetary data

cols = ['usd_price', 'valor', 'brl_price']
costs[cols] = costs[cols].apply(pd.to_numeric, errors='coerce')

costs['brl_price'] = costs['brl_price'].round(2)

  # Checagem da conversão
  # Conversion check
print(f'Column new type: {costs[['usd_price', 'valor', 'brl_price']].dtypes}')

Column new type: usd_price    float64
valor        float64
brl_price    float64
dtype: object


In [ ]:
# 8. Criação da coluna de ID único
# 8. Creating a unique ID column

costs.insert(0, 'purchase_id', range(1, len(costs) + 1))

In [ ]:
# 9. Conferindo colunas com valores nulos
# 9. Checking columns for null values

  # Os valores nulos correspondem a datas sem vínculo de conversão com a tabela de câmbio.
  # Este registro será irrelevante para as análises futuras. Portanto, nenhuma remoção é necessária.

  # Null values ​​correspond to dates with no conversion link to the exchange rate table.
  # This record will be irrelevant for future analyses. Therefore, no removal is required.

costs.isna().sum()

,0
purchase_id,0
product_id,0
product_name,0
category,0
start_date,0
usd_price,0
data,1
valor,1
brl_price,1


In [ ]:
# 10. Conferindo a quantidade de duplicatas
# 10. Checking for duplicate records

print(f'Dataframe total rows: {len(costs)}')
print(f'Total duplicates: {costs.duplicated().sum()}')

Dataframe total rows: 1260
Total duplicates: 0


In [ ]:
# 11. Renomeação de colunas
# 11. Column renaming

costs = costs.drop(['product_name', 'category'], axis=1)

costs = costs.rename(columns={

    # Formato [Entidade]_[Atributo] para adequação ao padrão de mercado
    # [Entity]_[Attribute] format for market standard alignment

    'start_date': 'purchase_date',
    'usd_price': 'unit_cost_USD',
    'brl_price': 'unit_cost_BRL',
    'data': 'exchange_date',
    'valor': 'exchange_rate'
})

costs.head()

,purchase_id,product_id,purchase_date,unit_cost_USD,exchange_date,exchange_rate,unit_cost_BRL
0,1,60,2016-01-02,27202.05,NaT,NaN,NaN
1,2,143,2016-01-06,1228.25,2016-01-06,4.0303,4950.22
2,3,130,2016-01-11,1293.13,2016-01-11,4.0153,5192.30
3,4,31,2016-01-15,2050.48,2016-01-15,4.0402,8284.35
4,5,105,2016-01-21,74.49,2016-01-21,4.1558,309.57


### Tabela de Relatório Financeiro (Financial Report Table)

In [ ]:
# 1. Criação daS colunas de custo e lucro
# 1. Cost and profit columns creation

sales_base = sales.sort_values('sale_date')
costs_base = costs.sort_values('purchase_date')

financial_report = pd.merge_asof(
    sales_base[['sale_id', 'product_qty', 'sale_date', 'sale_total_BRL']],
    costs_base[['purchase_id', 'purchase_date', 'unit_cost_BRL']],
    left_on='sale_date',
    right_on='purchase_date',
    direction='backward'
)


financial_report['total_cost_BRL'] = financial_report['unit_cost_BRL'] * financial_report['product_qty']

financial_report['gross_profit_BRL'] = financial_report['sale_total_BRL'] - financial_report['total_cost_BRL']

financial_report.head()

,sale_id,product_qty,sale_date,sale_total_BRL,purchase_id,purchase_date,unit_cost_BRL,total_cost_BRL,gross_profit_BRL
0,666,5,2023-01-01,132524.05,868,2022-12-30,8735.58,43677.9,88846.15
1,7765,2,2023-01-01,74960.70,868,2022-12-30,8735.58,17471.16,57489.54
2,5256,12,2023-01-01,1461139.00,868,2022-12-30,8735.58,104826.96,1356312.04
3,4997,1,2023-01-01,1893.00,868,2022-12-30,8735.58,8735.58,-6842.58
4,4294,5,2023-01-01,51332.30,868,2022-12-30,8735.58,43677.9,7654.4


In [ ]:
# 2. Conversão dos valores monetários
# 2. Converting monetary values

cols = ['total_cost_BRL', 'gross_profit_BRL']
financial_report[cols] = financial_report[cols].apply(pd.to_numeric, errors='coerce')

  # Checagem da conversão
  # Conversion check
print(f'Columns new type: {financial_report[['total_cost_BRL', 'gross_profit_BRL']].dtypes}')

Columns new type: total_cost_BRL      Float64
gross_profit_BRL    Float64
dtype: object


In [ ]:
# 3. Conferindo colunas com valores nulos
# 3. Checking columns for null values

financial_report.isna().sum()

,0
sale_id,5
product_qty,0
sale_date,0
sale_total_BRL,0
purchase_id,0
purchase_date,0
unit_cost_BRL,0
total_cost_BRL,0
gross_profit_BRL,0


In [ ]:
# 4. Conferindo a quantidade de duplicatas
# 4. Checking for Duplicate records

  # Os valores duplicados correspondem aos registros de datas sem vendas.
  # Esses registros serão utilizados futuramente para o cálculo da média de vendas, portanto, não é necessário a sua remoção.

  # The duplicate values ​​correspond to date records with no sales.
  # These records will be used in the future to calculate the sales average; therefore, their removal is not necessary.

print(f'Dataframe total rows: {len(financial_report)}')
print(f'Total duplicates: {financial_report.duplicated().sum()}')
print(f'Total sale_id duplicates: {financial_report.duplicated(subset=['sale_id']).sum()}')

duplicates_fr = financial_report[financial_report.duplicated(subset=['sale_id'], keep='first')]
display(duplicates_fr)

Dataframe total rows: 9900
Total duplicates: 0


In [ ]:
# 5. Reorganização das colunas
# 5. Column reordering

financial_report = financial_report.drop(['product_qty','sale_date', 'purchase_date', 'unit_cost_BRL'], axis=1)

cols = list(financial_report.columns)
cols.insert(1, cols.pop(cols.index('purchase_id')))

financial_report = financial_report[cols]

financial_report.head()

,sale_id,purchase_id,sale_total_BRL,total_cost_BRL,gross_profit_BRL
0,666,868,132524.05,43677.9,88846.15
1,7765,868,74960.70,17471.16,57489.54
2,5256,868,1461139.00,104826.96,1356312.04
3,4997,868,1893.00,8735.58,-6842.58
4,4294,868,51332.30,43677.9,7654.4


### Exportação das Tabelas (Tables export)

In [ ]:
# Conversão das tabelas para CSV
# Converting Tables to CSV

export_settings = {
    'index': False,
    'sep': ';',
    'encoding': 'utf-8-sig',
    'date_format': '%Y-%m-%d'
}

sales.to_csv('fact_sales.csv', **export_settings)
customers.to_csv('dim_customers.csv', **export_settings)
products.to_csv('dim_products.csv', **export_settings)
costs.to_csv('aux_costs.csv', **export_settings)
financial_report.to_csv('aux_financial_summary.csv', **export_settings)

In [ ]:
# Download dos arquivos CSV
# CSV files download

from google.colab import files

csvs = ['fact_sales.csv', 'dim_customers.csv', 'dim_products.csv', 'aux_costs.csv', 'aux_financial_summary.csv']

for csv in csvs:
    files.download(csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>